# Probe HM3D and ScanNet scenes

Both datasets represent **3D indoor scenes**, but the images mean slightly different things:

- **ScanNet** was captured by moving a real RGB-D camera. Its ordered RGB frames, depth maps, intrinsics, and camera poses form a trajectory through a reconstructed scene. Playing frames in order is approximately a walkthrough.
- **HM3D** is a collection of textured 3D meshes. In this project, the images are rendered/sampled views produced from those meshes. They belong to one 3D scene, but their filename order is not necessarily a smooth video trajectory.

This notebook inventories both datasets, shows evenly sampled views, creates an inline animation, and plots camera centers/directions whenever pose metadata is present. Run the cells from top to bottom.

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from PIL import Image

# Works when the notebook is launched from either the project root or code/.
here = Path.cwd().resolve()
PROJECT_ROOT = here if (here / 'dataset').exists() else here.parent
DATA_ROOT = PROJECT_ROOT / 'dataset' / 'frames'
DATASETS = {'HM3D': DATA_ROOT / 'hm3d-v0', 'ScanNet': DATA_ROOT / 'scannet-v0'}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

print('Project root:', PROJECT_ROOT)
for name, path in DATASETS.items():
    print(f'{name:7s}: {path}  (exists={path.exists()})')

## 1. Inventory the datasets

A *scene* is one folder such as `002-scannet-scene0709_00` or `000-hm3d-BFRyYbPCCPE`. The helper below avoids recursively scanning the entire dataset until a scene is selected.

In [ ]:
def scene_dirs(dataset_root):
    if not dataset_root.exists():
        return []
    try:
        return sorted((p for p in dataset_root.iterdir() if p.is_dir()), key=lambda p: p.name.lower())
    except PermissionError as exc:
        print(f'Cannot read {dataset_root}: {exc}')
        return []

scenes = {name: scene_dirs(root) for name, root in DATASETS.items()}
for name, paths in scenes.items():
    print(f'\n{name}: {len(paths)} scenes')
    for path in paths[:8]:
        print('  ', path.name)
    if len(paths) > 8:
        print(f'   ... {len(paths) - 8} more')

## 2. Select one scene from each dataset

Change either index to inspect another scene. The next cell prints a shallow tree so you can understand the available modalities before viewing them.

In [ ]:
HM3D_SCENE_INDEX = 0
SCANNET_SCENE_INDEX = 0

selected = {
    'HM3D': scenes['HM3D'][HM3D_SCENE_INDEX] if scenes['HM3D'] else None,
    'ScanNet': scenes['ScanNet'][SCANNET_SCENE_INDEX] if scenes['ScanNet'] else None,
}

def shallow_tree(root, max_entries=80):
    if root is None:
        return ['<no scene found>']
    lines, count = [], 0
    try:
        for path in sorted(root.rglob('*'), key=lambda p: str(p).lower()):
            rel = path.relative_to(root)
            if len(rel.parts) > 2:
                continue
            lines.append('  ' * (len(rel.parts) - 1) + ('[D] ' if path.is_dir() else '[F] ') + path.name)
            count += 1
            if count >= max_entries:
                lines.append('... output truncated')
                break
    except PermissionError as exc:
        lines.append(f'<permission denied: {exc}>')
    return lines

for name, scene in selected.items():
    print(f'\n{name}: {scene}')
    print('\n'.join(shallow_tree(scene)))

## 3. Find RGB frames and show a contact sheet

The discovery rule prefers paths containing `rgb` or `color` and excludes depth, masks, and semantic/instance images. If a scene uses an unusual layout, the printed candidate directories make it easy to adjust the keywords.

In [ ]:
def natural_key(path):
    return [int(s) if s.isdigit() else s.lower() for s in re.split(r'(\d+)', str(path))]

def find_rgb_images(scene):
    if scene is None:
        return []
    try:
        images = [p for p in scene.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    except PermissionError as exc:
        print(f'Cannot scan {scene}: {exc}')
        return []
    excluded = ('depth', 'mask', 'semantic', 'instance', 'label', 'normal')
    preferred = [p for p in images if any(k in str(p.parent).lower() for k in ('rgb', 'color'))
                 and not any(k in str(p).lower() for k in excluded)]
    fallback = [p for p in images if not any(k in str(p).lower() for k in excluded)]
    return sorted(preferred or fallback, key=natural_key)

rgb_frames = {name: find_rgb_images(scene) for name, scene in selected.items()}
for name, frames in rgb_frames.items():
    parents = sorted({str(p.parent.relative_to(selected[name])) for p in frames}) if selected[name] else []
    print(f'{name}: {len(frames)} RGB candidates; folders={parents[:6]}')

def show_sequence(frames, title, count=12):
    if not frames:
        print(f'No RGB images found for {title}.')
        return
    indices = np.linspace(0, len(frames) - 1, min(count, len(frames)), dtype=int)
    cols = min(4, len(indices)); rows = int(np.ceil(len(indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, idx in zip(axes.flat, indices):
        with Image.open(frames[idx]) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'frame {idx}: {frames[idx].name}', fontsize=9)
    fig.suptitle(f'{title}: evenly sampled views ({len(frames)} total)', fontsize=14)
    plt.tight_layout()
    plt.show()

for name, frames in rgb_frames.items():
    show_sequence(frames, name)

## 4. Play the views inline

This uses a JavaScript-backed Matplotlib animation, so it does not write a GIF file or require ImageMagick. `MAX_ANIMATION_FRAMES` limits notebook size. For ScanNet the order is a captured camera sequence; for HM3D, treat it as an ordered slideshow unless the sampling pipeline explicitly generated a trajectory.

In [ ]:
MAX_ANIMATION_FRAMES = 100
FRAME_INTERVAL_MS = 120

def animate_scene(frames, title, max_frames=MAX_ANIMATION_FRAMES, interval=FRAME_INTERVAL_MS):
    if not frames:
        print(f'No frames to animate for {title}.')
        return None
    take = np.linspace(0, len(frames) - 1, min(max_frames, len(frames)), dtype=int)
    chosen = [frames[i] for i in take]
    with Image.open(chosen[0]) as first:
        first_array = np.asarray(first.convert('RGB'))
    fig, ax = plt.subplots(figsize=(8, 5))
    artist = ax.imshow(first_array)
    label = ax.set_title('')
    ax.axis('off')

    def update(i):
        with Image.open(chosen[i]) as im:
            artist.set_data(np.asarray(im.convert('RGB')))
        label.set_text(f'{title} — sampled frame {i + 1}/{len(chosen)} — {chosen[i].name}')
        return artist, label

    animation = FuncAnimation(fig, update, frames=len(chosen), interval=interval, blit=False)
    plt.close(fig)
    return HTML(animation.to_jshtml(default_mode='loop'))

# Change this to 'HM3D' or 'ScanNet'.
ANIMATE_DATASET = 'ScanNet'
animation_html = animate_scene(rgb_frames[ANIMATE_DATASET], ANIMATE_DATASET)
if animation_html is not None:
    display(animation_html)

## 5. Load and plot camera poses (optional metadata)

**The direct OpenEQA/Hugging Face archives downloaded by this project are RGB-only, so finding zero poses is expected.** They contain episode image histories, not the full raw HM3D or ScanNet 3D data. This section becomes useful if the folders are later replaced or augmented with frames extracted from raw scenes using OpenEQA's non-`--rgb-only` workflow.

A pose is normally a 4×4 camera-to-world transform. Its last column is the camera center; its rotation columns describe camera orientation. This loader supports one text matrix per frame plus common `.npy`/`.npz` pose arrays. It rejects non-finite and malformed matrices.

Coordinate conventions vary. The plot tries both common camera forward axes (`-Z` and `+Z`); use `FORWARD_SIGN` to flip the arrows if they point backward. The trajectory itself (camera centers) is unaffected.

In [ ]:
def as_pose_stack(array):
    arr = np.asarray(array, dtype=float)
    if arr.shape == (4, 4):
        arr = arr[None, ...]
    elif arr.ndim == 2 and arr.shape[1] == 16:
        arr = arr.reshape(-1, 4, 4)
    if arr.ndim != 3 or arr.shape[1:] != (4, 4):
        return np.empty((0, 4, 4))
    return arr[np.isfinite(arr).all(axis=(1, 2))]

def find_poses(scene):
    if scene is None:
        return np.empty((0, 4, 4)), []
    try:
        # OpenEQA writes flat poses as 00000.txt (no 'pose' directory/name).
        files = [p for p in scene.rglob('*') if p.is_file() and (
            'pose' in str(p).lower() or p.suffix.lower() in {'.txt', '.pose', '.npy', '.npz'}
        ) and not any(k in p.name.lower() for k in ('intrinsic', 'extrinsic'))]
    except PermissionError as exc:
        print(f'Cannot scan poses in {scene}: {exc}')
        return np.empty((0, 4, 4)), []
    files = sorted(files, key=natural_key)
    text_files = [p for p in files if p.suffix.lower() in {'.txt', '.pose'}]
    matrices, sources = [], []
    for path in text_files:
        try:
            pose = as_pose_stack(np.loadtxt(path))
            if len(pose):
                matrices.append(pose[0]); sources.append(path)
        except (ValueError, OSError):
            pass
    if matrices:
        return np.stack(matrices), sources
    for path in files:
        try:
            if path.suffix.lower() == '.npy':
                poses = as_pose_stack(np.load(path, allow_pickle=False))
            elif path.suffix.lower() == '.npz':
                archive = np.load(path, allow_pickle=False)
                poses = next((as_pose_stack(archive[k]) for k in archive.files
                              if len(as_pose_stack(archive[k]))), np.empty((0, 4, 4)))
            else:
                continue
            if len(poses):
                return poses, [path] * len(poses)
        except (ValueError, OSError):
            pass
    return np.empty((0, 4, 4)), []

poses = {}
for name, scene in selected.items():
    poses[name], pose_sources = find_poses(scene)
    source_hint = pose_sources[0] if pose_sources else '<none found>'
    if len(poses[name]):
        print(f'{name}: {len(poses[name])} valid poses; first source: {source_hint}')
    else:
        print(f'{name}: RGB-only episode archive — no pose files included (expected for this download).')

In [ ]:
FORWARD_SIGN = -1  # change to +1 if the dataset uses +Z as camera-forward
POSE_ARROW_COUNT = 30

def set_axes_equal(ax, points):
    center = points.mean(axis=0)
    radius = max(np.ptp(points, axis=0).max() / 2, 1e-3)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)

def plot_camera_poses(pose_stack, title, forward_sign=FORWARD_SIGN):
    if not len(pose_stack):
        print(f'{title}: pose plot skipped because this RGB-only archive contains no camera transforms.')
        return
    centers = pose_stack[:, :3, 3]
    forward = forward_sign * pose_stack[:, :3, 2]
    chosen = np.linspace(0, len(pose_stack) - 1, min(POSE_ARROW_COUNT, len(pose_stack)), dtype=int)
    scale = max(np.ptp(centers, axis=0).max() * 0.06, 0.05)
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(*centers.T, color='steelblue', linewidth=1, alpha=.75, label='camera path')
    ax.scatter(*centers.T, c=np.arange(len(centers)), cmap='viridis', s=10, label='time/order')
    ax.quiver(*centers[chosen].T, *forward[chosen].T, length=scale, normalize=True, color='crimson')
    ax.scatter(*centers[0], color='limegreen', s=80, label='start')
    ax.scatter(*centers[-1], color='black', s=80, marker='x', label='end')
    ax.set(xlabel='world X', ylabel='world Y', zlabel='world Z', title=f'{title}: camera trajectory ({len(centers)} poses)')
    set_axes_equal(ax, centers)
    ax.legend()
    plt.show()

for name, pose_stack in poses.items():
    plot_camera_poses(pose_stack, name)

## How to interpret the result

A single image is only one 2D observation. Depth tells how far visible surfaces are from that camera; the pose places that observation in the scene's shared 3D coordinate system. Many overlapping RGB + depth + pose observations allow a system such as ConceptGraphs to associate objects across views and build a 3D scene graph.

Important caveats:

1. Seeing every stored image is not the same as seeing every surface—occlusion and incomplete coverage remain.
2. ScanNet frame order generally has temporal meaning. HM3D render order only has motion meaning if the renderer sampled a continuous agent trajectory.
3. Pose matrices may be camera-to-world or world-to-camera. This notebook assumes camera-to-world, the usual extracted ScanNet/ConceptGraphs convention. If a trajectory looks implausible, check the dataset exporter before inverting matrices.
4. For a true 3D inspection, the mesh or fused point cloud is more informative than a GIF; this notebook focuses on understanding the frame and pose data that downstream models consume.